In [1]:
# %pip install -U ultralytics

In [ ]:
# Import of all packages
import ultralytics
from ultralytics import YOLO
import shutil
import zipfile
from pathlib import Path
import yaml
from google.colab import drive

drive.mount("/content/drive")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


In [ ]:
# Here are done some configurations and openning of files

DRIVE_ZIP = Path(
    "/content/drive/MyDrive/DLE/taco_yolo.zip"
)
LOCAL_ZIP = Path("/content/taco_yolo.zip")
DATASET_DIR = Path("/content/taco_yolo")

shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)

with zipfile.ZipFile(LOCAL_ZIP) as archive:
    archive.extractall("/content")

DATA_YAML = DATASET_DIR / "data.yaml"

with DATA_YAML.open(encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

dataset_config["path"] = str(DATASET_DIR)

with DATA_YAML.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        dataset_config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

Copying ZIP from Drive...
Extracting locally...
Dataset extracted to: /content/taco_yolo


In [ ]:
# The configuration of images for first two trainings were with train/val/test of 80/10/10(1200/150/150)
# For further training, I want to try 4-fold cross-validation, doing only train/test of 90/10(1350/150)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

for split in ("train", "test"):
    image_directory = DATASET_DIR / "images" / split
    label_directory = DATASET_DIR / "labels" / split

    images = [
        path
        for path in image_directory.rglob("*")
        if path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(label_directory.rglob("*.txt"))

    print(
        f"{split:5s}: "
        f"{len(images):4d} images, "
        f"{len(labels):4d} label files"
    )

train: 1200 images, 1200 label files
test :  150 images,  150 label files


In [ ]:
# Training of the model with different parameters
DRIVE_RUNS = Path(
    "/content/drive/MyDrive/YOLO/runs"
)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

model = YOLO("yolo26n.pt")

training_results = model.train(
    data=str(DATA_YAML),
    epochs=1,
    imgsz=960,
    batch=8,
    # device=0,
    patience=30,
    optimizer="auto",
    amp=True,
    workers=2,
    cache=False,
    seed=42,
    plots=True,
    save=True,
    save_period=10,
    project=str(DRIVE_RUNS),
    name="taco10_yolo26n",
)

RUN_DIRECTORY = Path(model.trainer.save_dir)
print("Results saved to:", RUN_DIRECTORY)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/taco_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=taco10_yolo26n-3, nbs=64, nms=Fals

In [ ]:
# First training
# epochs=100,
# imgsz=640,

RUN_DIRECTORY = Path(
    "/content/drive/MyDrive/YOLO/runs/taco10_yolo26n"
)

BEST_WEIGHTS = RUN_DIRECTORY / "weights" / "best.pt"

best_model = YOLO(str(BEST_WEIGHTS))

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
)

print("mAP50:", test_metrics.box.map50)
print("mAP50–95:", test_metrics.box.map)

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 189.6±58.6 MB/s, size: 1811.1 KB)
val: Scanning /content/taco_yolo/labels/test/batch_1... 150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 404.2it/s 0.4s
val: New cache created: /content/taco_yolo/labels/test/batch_1.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.1it/s 9.0s
                   all        150        485      0.298      0.233      0.209      0.149
                 Other         69        159      0.218      0.258      0.154      0.101
                Bottle         39         54      0.511      0.463      0.443      0.347
            Bottle cap         28         32       0.59       0.27      0.295      0.161
                   Can         20         26      0.303      0

In [ ]:
# Second training (we can see a small improvement)
# epochs=50,
# imgsz=960,

RUN_DIRECTORY2 = Path(
    "/content/drive/MyDrive/YOLO/runs/taco10_yolo26n-2"
)

BEST_WEIGHTS2 = RUN_DIRECTORY2 / "weights" / "best.pt"

best_model2 = YOLO(str(BEST_WEIGHTS2))

test_metrics2 = best_model2.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=960,
    device=0,
)

print("mAP50:", test_metrics2.box.map50)
print("mAP50–95:", test_metrics2.box.map)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 174.6±45.2 MB/s, size: 1191.6 KB)
val: Scanning /content/taco_yolo/labels/test/batch_1... 150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 431.5it/s 0.3s
val: New cache created: /content/taco_yolo/labels/test/batch_1.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.1s/it 10.6s
                   all        150        485      0.359       0.28      0.215      0.154
                 Other         69        159      0.264      0.302      0.182      0.128
                Bottle         39         54      0.481      0.537      0.464      0.333
            Bottle cap         28         32      0.481       0.29      0.359       0.24
                   Can         20         26      0.295      